<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_21_encapsulation_scope/note_lesson_21_encapsulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 21 — Інкапсуляція, область видимості: замовлення, яке себе захищає

Бухгалтер знайшла в звіті від'ємний виторг: скрипт знижок писав прямо в `order.bill`. А ще виставляв статус `"delivered"` замовленням, які кухня й не починала. Перевірка в `__init__` захищає лише **народження** об'єкта. Сьогодні — як зробити, щоб у стан вели тільки **правильні двері**.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія, діаграма автомата статусів і архітектура «інтерфейс проти стану» — у книзі: [Урок 21. Інкапсуляція, область видимості](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_21/).

## 🔁 Пригадай (без підглядання)

1. Коли виконується перевірка в `__init__`?
2. Навіщо `nonlocal`?
3. Що таке принцип підстановки Лісков?

<details>
<summary>Відповіді</summary>

1. Один раз — при створенні об'єкта.
2. Щоб змінити змінну оточуючої функції.
3. Нащадок має працювати всюди, де очікують батька.

</details>

## 1. Як зламати інваріант

**Прогноз:** що надрукує клітинка?

In [ ]:
class Order:
    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self.bill = bill


order = Order(1, 540.0)
order.bill = order.bill - 600
print(order.bill)

<details>
<summary>Відповідь</summary>

`-60.0` — без жодної помилки: `bill` відкритий, писати в нього може будь-хто.

</details>

## 2. `_name` і `__name`

**Прогноз:** як виглядатиме `order.__dict__`?

In [ ]:
class Order:
    def __init__(self, order_id, bill):
        self.id = order_id
        self._bill = bill
        self.__kitchen_note = "без цибулі"


order = Order(1, 540.0)
print(order.__dict__)
print(order._bill)

try:
    print(order.__kitchen_note)
except AttributeError as error:
    print("AttributeError:", error)

<details>
<summary>Відповідь</summary>

`__kitchen_note` став `_Order__kitchen_note` — name mangling. `_bill` — звичайний атрибут, підкреслення лише домовленість.

</details>

### Колізія імен у нащадках

**Прогноз:** що поверне `tracked.status()`?

In [ ]:
class Delivery:
    def __init__(self, order_id):
        self.order_id = order_id
        self._status = "нова"

    def status(self):
        return self._status


class TrackedDelivery(Delivery):
    def __init__(self, order_id, gps):
        super().__init__(order_id)
        self._status = f"GPS {gps}"


tracked = TrackedDelivery(7, "50.45,30.52")
print(tracked.status())

<details>
<summary>Відповідь</summary>

`GPS 50.45,30.52` — нащадок затер `_status` батька. Виправлення — `__status`: у кожного класу своє поле.

</details>

In [ ]:
class Delivery:
    def __init__(self, order_id):
        self.order_id = order_id
        self.__status = "нова"

    def status(self):
        return self.__status


class TrackedDelivery(Delivery):
    def __init__(self, order_id, gps):
        super().__init__(order_id)
        self.__status = f"GPS {gps}"


tracked = TrackedDelivery(7, "50.45,30.52")
print(tracked.status())
print(tracked.__dict__)

## 3. Методи — двері в стан

In [ ]:
class Order:
    MAX_DISCOUNT = 50

    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self._bill = bill

    def bill(self):
        return self._bill

    def apply_discount(self, percent):
        if not 0 < percent <= self.MAX_DISCOUNT:
            raise ValueError(f"знижка має бути від 1 до {self.MAX_DISCOUNT} %, а маємо {percent}")
        self._bill = round(self._bill * (100 - percent) / 100, 2)


order = Order(1, 540.0)
order.apply_discount(10)
print(order.bill())

try:
    order.apply_discount(111)
except ValueError as error:
    print(error)
print(order.bill())

## 4. Статуси: скінченний автомат

`new → cooking → on_the_way → delivered`; `cancelled` — лише з `new` чи `cooking`. Словник переходів і один метод `_move`.

In [ ]:
class Order:
    TRANSITIONS = {
        "new": {"cooking", "cancelled"},
        "cooking": {"on_the_way", "cancelled"},
        "on_the_way": {"delivered"},
        "delivered": set(),
        "cancelled": set(),
    }

    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self._bill = bill
        self._status = "new"
        self._history = ["new"]

    def _move(self, new_status):
        if new_status not in self.TRANSITIONS[self._status]:
            raise ValueError(f"замовлення №{self.id}: не можна {self._status} → {new_status}")
        self._status = new_status
        self._history.append(new_status)

    def cook(self):
        self._move("cooking")

    def send(self):
        self._move("on_the_way")

    def deliver(self):
        self._move("delivered")

    def cancel(self):
        self._move("cancelled")


order = Order(1, 540.0)
order.cook()
order.send()
try:
    order.cancel()
except ValueError as error:
    print(error)
order.deliver()
print(order._history)

## 5. `@property`: читати — так, писати — лише через правила

In [ ]:
class Order(Order):
    @property
    def status(self):
        return self._status

    @property
    def history(self):
        return tuple(self._history)


order = Order(2, 320.0)
order.cook()
print(order.status, order.history)

try:
    order.status = "delivered"
except AttributeError as error:
    print(type(error).__name__)

In [ ]:
class Order(Order):
    @property
    def bill(self):
        return self._bill

    @bill.setter
    def bill(self, value):
        if value <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {value}")
        self._bill = value


order = Order(3, 980.0)
order.bill = 900.0
try:
    order.bill = order.bill - 1000
except ValueError as error:
    print(error)
print(order.bill)

## 🛠 Вправа 1. Повернення замовлення

Додай у `SafeOrder` (клітинка нижче) стан `"returned"`: перехід лише з `"delivered"`, метод `return_order(reason)` зберігає причину в `_return_reason`, властивість `return_reason` її читає; порожня причина → `ValueError`.

In [ ]:
class SafeOrder:
    TRANSITIONS = Order.TRANSITIONS
    MAX_DISCOUNT = 50

    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self._bill = bill
        self._status = "new"
        self._history = ["new"]

    @property
    def bill(self):
        return self._bill

    @property
    def status(self):
        return self._status

    @property
    def history(self):
        return tuple(self._history)

    def apply_discount(self, percent):
        if self._status != "new":
            raise ValueError(f"знижку можна дати лише новому замовленню, а статус {self._status}")
        if not 0 < percent <= self.MAX_DISCOUNT:
            raise ValueError(f"знижка має бути від 1 до {self.MAX_DISCOUNT} %, а маємо {percent}")
        self._bill = round(self._bill * (100 - percent) / 100, 2)

    def _move(self, new_status):
        if new_status not in self.TRANSITIONS[self._status]:
            raise ValueError(f"замовлення №{self.id}: не можна {self._status} → {new_status}")
        self._status = new_status
        self._history.append(new_status)

    def cook(self):
        self._move("cooking")

    def send(self):
        self._move("on_the_way")

    def deliver(self):
        self._move("delivered")

    def cancel(self):
        self._move("cancelled")


order = SafeOrder(1, 540.0)
order.apply_discount(10)
order.cook()
for action in (lambda: order.apply_discount(5), order.deliver, lambda: setattr(order, "bill", 1.0)):
    try:
        action()
    except (ValueError, AttributeError) as error:
        print(type(error).__name__, "—", error if isinstance(error, ValueError) else "запис заборонено")
order.send()
order.deliver()
print(order.bill, order.status, order.history)

In [ ]:
class ReturnableOrder(SafeOrder):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    TRANSITIONS = {**SafeOrder.TRANSITIONS, "delivered": {"returned"}, "returned": set()}

    def __init__(self, order_id, bill):
        super().__init__(order_id, bill)
        self._return_reason = None

    @property
    def return_reason(self):
        return self._return_reason

    def return_order(self, reason):
        if not reason.strip():
            raise ValueError("вкажи причину повернення")
        self._move("returned")
        self._return_reason = reason
    # END SOLUTION


order = ReturnableOrder(5, 700.0)
order.cook(); order.send(); order.deliver()
order.return_order("холодна піца")
assert order.status == "returned" and order.return_reason == "холодна піца"
assert order.history[-2:] == ("delivered", "returned")
for bad in (lambda: ReturnableOrder(6, 100.0).return_order("інше"),
            lambda: order.return_order("ще раз"),
            lambda: setattr(order, "return_reason", "інше")):
    try:
        bad()
    except (ValueError, AttributeError):
        pass
    else:
        raise AssertionError("очікували ValueError чи AttributeError")
try:
    fresh = ReturnableOrder(7, 100.0)
    fresh.cook(); fresh.send(); fresh.deliver()
    fresh.return_order("   ")
except ValueError:
    pass
else:
    raise AssertionError("порожня причина — ValueError")
print("✅ Вправа 1 пройдена")

## 6. Область видимості: `global` і спільний лічильник

**Прогноз:** які номери отримають перші замовлення Подолу й Оболоні з глобальним лічильником?

In [ ]:
total_orders = 0


def add_order_broken():
    total_orders += 1


try:
    add_order_broken()
except UnboundLocalError as error:
    print(type(error).__name__)


def add_order():
    global total_orders
    total_orders += 1
    return total_orders


add_order()
add_order()
print(total_orders)


podil_first = add_order()
obolon_first = add_order()
print(podil_first, obolon_first)

<details>
<summary>Відповідь</summary>

`3 4`: лічильник один на весь модуль, і відділення рахують у ту саму змінну. Рішення — стан в об'єкті:

</details>

In [ ]:
class Branch:
    def __init__(self, name):
        self.name = name
        self._next_id = 1

    def add_order(self):
        order_id = self._next_id
        self._next_id += 1
        return order_id


podil, obolon = Branch("Поділ"), Branch("Оболонь")
print(podil.add_order(), podil.add_order(), obolon.add_order())

## 🛠 Вправа 2. Захищений промокод

`Promo(code, percent, uses)`: лічильник у `__left`, властивість `left` лише для читання, `apply(bill)` — єдині двері. Потім `VipPromo(Promo)` зі своїм полем `__left`, яке не зачіпає батьківське.

In [ ]:
class Promo:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def __init__(self, code, percent, uses):
        self.code = code
        self.percent = percent
        self.__left = uses

    @property
    def left(self):
        return self.__left

    def apply(self, bill):
        if self.__left == 0:
            raise ValueError(f"промокод {self.code} вичерпано")
        self.__left -= 1
        return round(bill * (100 - self.percent) / 100, 2)
    # END SOLUTION


class VipPromo(Promo):
    def __init__(self, code, percent, uses, bonus_uses):
        super().__init__(code, percent, uses)
        self.__left = bonus_uses          # своє поле нащадка


lucky = Promo("LUCKY20", 20, 2)
assert lucky.apply(500.0) == 400.0 and lucky.left == 1
try:
    lucky.left = 999
except AttributeError:
    pass
else:
    raise AssertionError("left — лише для читання")
assert "_Promo__left" in lucky.__dict__
vip = VipPromo("VIP30", 30, 3, 10)
assert vip.left == 3 and vip._VipPromo__left == 10, "поле нащадка не затерло батьківське"
print("✅ Вправа 2 пройдена")

## 🐛 Вправа 3. Знайди помилку

`history` повертає сам список — і автомат статусів можна оминути. Виправ властивість.

In [ ]:
class LeakyOrder(SafeOrder):
    @property
    def history(self):
        return self._history              # ← у чому проблема?
# BEGIN SOLUTION
class LeakyOrder(SafeOrder):
    @property
    def history(self):
        return tuple(self._history)
# END SOLUTION


order = LeakyOrder(9, 300.0)
try:
    order.history.append("delivered")
except AttributeError:
    pass
assert order.history == ("new",) and order.status == "new"
print("✅ Вправа 3 пройдена: історію не можна змінити ззовні")

## ✅ Самоперевірка

1. Чому перевірки в `__init__` недостатньо?
2. Чим `_bill` відрізняється від `__bill`?
3. Навіщо автомату словник `TRANSITIONS` і один `_move`?
4. Чому `history` повертає кортеж?
5. Де жити правилу «одна доставка на замовлення»?

<details>
<summary>Відповіді</summary>

1. `__init__` виконується один раз; далі відкритий атрибут змінює будь-хто.
2. `_bill` — домовленість; `__bill` → `_Order__bill` (name mangling).
3. Правила — дані в одному місці, перевірка — в одному методі.
4. Список можна змінити ззовні.
5. У сервісі: це зв'язок між різними об'єктами.

</details>

### Шпаргалка

```python
self._bill                # внутрішнє — домовленість
self.__left               # → self._Promo__left, захист від колізій у нащадках

@property
def status(self):         # order.status — читання без дужок
    return self._status   # без сеттера — AttributeError при записі

@bill.setter
def bill(self, value):    # order.bill = ... — з перевіркою
    ...

TRANSITIONS = {"new": {"cooking", "cancelled"}, ...}   # автомат
def _move(self, new): ... # єдине місце зміни статусу

# область видимості: локальна → атрибут об'єкта → модуль (лише константи); global — майже ніколи
```

## Далі

- Практикум — [`lab_lesson_21_cars_oop.ipynb`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_21_encapsulation_scope/lab_lesson_21_cars_oop.ipynb): «Автомобілі як об'єкти».
- **Урок 22 — практикум П4**: рекурсія, «розділяй і володарюй», перебір з поверненням.